# Validate: `_apply_entry_offset`

Checks that `_apply_entry_offset(signal_df, k)` correctly delays each trade's
entry by `k` bars and drops trades that don't survive the offset.

**Rule:** for each contiguous in-trade run (consecutive bars with `cycle != "None"`):

- Run length `≤ k + 1`: reset entirely to flat.
- Otherwise: bars `0..k-1` of the run become flat, bar `k` becomes the new
  entry (`cycle="init"`, `enter=1`, `signal_open=0`); the original held / exit
  bars are left untouched.

Used by `BarBacktest.run(signal_df, entry_offset=k)`.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hailmary.backtest.signal_backtest import _apply_entry_offset
from hailmary.viz.theme import PALETTE, apply_theme

# Cycle pattern shorthand for hand-built test frames:
#   _  flat / None
#   I  init  (enter=1, exit=0, signal_open=1)
#   H  held  (enter=0, exit=0, signal_open=1)
#   E  exit  (enter=0, exit=1, signal_open=0)
CYCLE_MAP = {
    "_": ("None", 0, 0, 0),
    "I": ("init", 1, 0, 1),
    "H": ("held", 0, 0, 1),
    "E": ("exit", 0, 1, 0),
}
INV_CYCLE = {"None": "_", "init": "I", "held": "H", "exit": "E"}


def make_signal_df(patterns: dict[str, str]) -> pd.DataFrame:
    """Build a (symbol, timestamp) signal_df from cycle pattern strings."""
    n = len(next(iter(patterns.values())))
    idx = pd.bdate_range("2024-01-01", periods=n, name="timestamp")
    frames = []
    for sym, pattern in patterns.items():
        rows = [CYCLE_MAP[c] for c in pattern]
        df = pd.DataFrame(rows, columns=["cycle", "enter", "exit", "signal_open"], index=idx)
        df["symbol"] = sym
        frames.append(df.reset_index().set_index(["symbol", "timestamp"]))
    return pd.concat(frames).sort_index()


def cycle_string(df: pd.DataFrame, symbol: str) -> str:
    return "".join(INV_CYCLE[c] for c in df.loc[symbol, "cycle"])

## 1. Hand-Crafted Cases

Each case fully determines the expected output, so we assert character-by-character.

| # | Input pattern   | k | Expected output | Reason                                              |
|---|-----------------|---|-----------------|-----------------------------------------------------|
| 1 | `_IHHHE___`     | 0 | `_IHHHE___`     | k=0 is a no-op                                      |
| 2 | `_IHHHE___`     | 1 | `__IHHE___`     | run_len=5, drop 1 bar, new init at pos 1            |
| 3 | `_IHHHE___`     | 2 | `___IHE___`     | drop 2 bars, new init at pos 2                      |
| 4 | `_IHHHE___`     | 3 | `____IE___`     | drop 3 bars, new init at pos 3, single held bar gone |
| 5 | `_IHHHE___`     | 4 | `_________`     | run_len=5 ≤ k+1=5 → wipe entire run                  |
| 6 | `_IHHHE___`     | 5 | `_________`     | wipe                                                |
| 7 | `___IE____`     | 1 | `_________`     | run_len=2 ≤ 2 → wipe                                |
| 8 | `_IHHHHHHE`     | 3 | `____IHHHE`     | run_len=8 > 4, drop 3, new init at pos 3            |

In [2]:
cases = [
    ("k=0 noop",                  "_IHHHE___", 0, "_IHHHE___"),
    ("k=1, run_len=5",            "_IHHHE___", 1, "__IHHE___"),
    ("k=2, run_len=5",            "_IHHHE___", 2, "___IHE___"),
    ("k=3, run_len=5",            "_IHHHE___", 3, "____IE___"),
    ("k=4 wipes (run_len = k+1)", "_IHHHE___", 4, "_________"),
    ("k=5 wipes (run_len < k+1)", "_IHHHE___", 5, "_________"),
    ("Two-bar IE wiped",          "___IE____", 1, "_________"),
    ("Long run survives",         "_IHHHHHHE", 3, "____IHHHE"),
]

rows = []
all_pass = True
for name, pattern, k, expected in cases:
    df_in  = make_signal_df({"S": pattern})
    df_out = _apply_entry_offset(df_in, k)
    got = cycle_string(df_out, "S")
    ok  = got == expected
    all_pass = all_pass and ok
    rows.append({
        "Case":     name,
        "Input":    pattern,
        "k":        k,
        "Expected": expected,
        "Got":      got,
        "Pass":     "✓" if ok else "✗ FAIL",
    })

display(pd.DataFrame(rows).set_index("Case"))
print()
print("All cases pass" if all_pass else "*** FAILURES DETECTED ***")

,Input,k,Expected,Got,Pass
Case,,,,,
k=0 noop,_IHHHE___,0,_IHHHE___,_IHHHE___,✓
"k=1, run_len=5",_IHHHE___,1,__IHHE___,__IHHE___,✓
"k=2, run_len=5",_IHHHE___,2,___IHE___,___IHE___,✓
"k=3, run_len=5",_IHHHE___,3,____IE___,____IE___,✓
k=4 wipes (run_len = k+1),_IHHHE___,4,_________,_________,✓
k=5 wipes (run_len < k+1),_IHHHE___,5,_________,_________,✓
Two-bar IE wiped,___IE____,1,_________,_________,✓
Long run survives,_IHHHHHHE,3,____IHHHE,____IHHHE,✓



All cases pass


## 2. Visual Walkthrough

Show the cycle column before and after the offset for one case.  Each cell is
one bar; colour denotes cycle (`init` blue, `held` green, `exit` red, `None`
muted).

In [3]:
CYCLE_COLOR = {
    "None": PALETTE["border"],
    "init": PALETTE["accent_blue"],
    "held": PALETTE["accent_green"],
    "exit": PALETTE["accent_red"],
}


def plot_before_after(pattern: str, k: int, title: str) -> go.Figure:
    df_in  = make_signal_df({"S": pattern})
    df_out = _apply_entry_offset(df_in, k)
    n = len(pattern)

    fig = go.Figure()
    for label, df, y in [("original", df_in, 1), ("delayed", df_out, 0)]:
        cycles = df.loc["S", "cycle"].tolist()
        for i, cyc in enumerate(cycles):
            fig.add_shape(
                type="rect",
                x0=i - 0.45, x1=i + 0.45,
                y0=y - 0.4, y1=y + 0.4,
                line=dict(color=PALETTE["surface"], width=1),
                fillcolor=CYCLE_COLOR[cyc],
                opacity=0.85 if cyc != "None" else 0.4,
                layer="below",
            )
            fig.add_annotation(
                x=i, y=y, text=INV_CYCLE[cyc],
                showarrow=False,
                font=dict(color=PALETTE["text_primary"], size=12, family="monospace"),
            )

    # Legend dummies
    for cyc, color in CYCLE_COLOR.items():
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=14, color=color, symbol="square"),
            name=cyc,
        ))

    apply_theme(fig, title=f"{title} (k={k})", height=240)
    fig.update_layout(
        xaxis=dict(title_text="bar", range=[-0.6, n - 0.4], tickmode="array",
                   tickvals=list(range(n))),
        yaxis=dict(
            tickvals=[0, 1], ticktext=["delayed", "original"],
            range=[-0.6, 1.6], showgrid=False,
        ),
        showlegend=True,
    )
    return fig


plot_before_after("_IHHHHHE__", k=2, title="Long trade survives").show()
plot_before_after("_IHHE_____", k=2, title="Short trade wiped (run_len = k+1)").show()
plot_before_after("_IHHHE____", k=3, title="Tight survival (run_len = k+2)").show()

## 3. Back-to-Back Trades Regression

**Bug fixed 2026-05-04.** Using `cycle != "None"` transitions to detect run
boundaries failed when one trade's exit on day N was immediately followed by
another's init on day N+1 with no flat bar between — both got the same
`run_id`, so only the first trade received the offset shift.  The fix uses
`enter == 1` to mark new runs.

Below: two adjacent trades (`IHHE` then `IHE`).  With `k=1`, both should be
shifted by one bar.

In [5]:
regression_cases = [
    # Adjacent E→I transition: trade 1 IHHE then trade 2 IHE with one flat between
    ("E then flat then I, k=1", "_IHHE_IHE___", 1, "__IHE__IE___"),
    ("E then flat then I, k=2", "_IHHE_IHE___", 2, "___IE_______"),
    # Truly back-to-back: E on bar N, I on bar N+1 (no flat between)
    ("E→I no gap, k=1",         "IHHEIHE___",   1, "_IHE_IE___"),
    ("E→I no gap, k=2",         "IHHEIHE___",   2, "__IE______"),
]

rows = []
all_pass = True
for name, pattern, k, expected in regression_cases:
    df_in  = make_signal_df({"S": pattern})
    df_out = _apply_entry_offset(df_in, k)
    got = cycle_string(df_out, "S")
    ok  = got == expected
    all_pass = all_pass and ok
    rows.append({
        "Case":     name,
        "Input":    pattern,
        "k":        k,
        "Expected": expected,
        "Got":      got,
        "Pass":     "✓" if ok else "✗ FAIL",
    })

display(pd.DataFrame(rows).set_index("Case"))
print()
print("All regression cases pass" if all_pass else "*** REGRESSION DETECTED ***")

plot_before_after("_IHHE_IHE___", k=1, title="Back-to-back trades both shift").show()

,Input,k,Expected,Got,Pass
Case,,,,,
"E then flat then I, k=1",_IHHE_IHE___,1,__IHE__IE___,__IHE__IE___,✓
"E then flat then I, k=2",_IHHE_IHE___,2,___IE_______,___IE_______,✓
"E→I no gap, k=1",IHHEIHE___,1,_IHE_IE___,_IHE_IE___,✓
"E→I no gap, k=2",IHHEIHE___,2,__IE______,__IE______,✓



All regression cases pass


,Input,k,Expected,Got,Pass
Case,,,,,
"E then flat then I, k=1",_IHHE_IHE___,1,__IHE__IE___,__IHE__IE___,✓
"E then flat then I, k=2",_IHHE_IHE___,2,___IE_______,___IE_______,✓
"E→I no gap, k=1",IHHEIHE___,1,_IHE_IE___,_IHE_IE___,✓
"E→I no gap, k=2",IHHEIHE___,2,__IE______,__IE______,✓



All regression cases pass


## 4. Multi-Symbol Independence

Each symbol's runs are partitioned independently — a long trade on one symbol
must not affect a short trade on another.

In [8]:
patterns_in = {
    "AAA": "_IHHHHE___",   # run_len=6   → survives k=2
    "BBB": "_IHE______",   # run_len=3   → wiped at k=2
    "CCC": "_IHHE_IHE_",   # trade 1 run_len=4 survives, trade 2 run_len=3 wipes
}

expected_out = {
    "AAA": "___IHHE___",   # drop 2, new init at pos 2
    "BBB": "__________",   # wiped
    "CCC": "___IE_____",   # trade 1 shifts, trade 2 wipes
}

df_in  = make_signal_df(patterns_in)
df_out = _apply_entry_offset(df_in, 2)

rows = []
all_pass = True
for sym in patterns_in:
    got = cycle_string(df_out, sym)
    exp = expected_out[sym]
    ok  = got == exp
    all_pass = all_pass and ok
    rows.append({
        "Symbol":   sym,
        "Input":    patterns_in[sym],
        "Expected": exp,
        "Got":      got,
        "Pass":     "✓" if ok else "✗ FAIL",
    })

display(pd.DataFrame(rows).set_index("Symbol"))
print()
print("All multi-symbol cases pass" if all_pass else "*** FAILURES DETECTED ***")

,Input,Expected,Got,Pass
Symbol,,,,
AAA,_IHHHHE___,___IHE____,___IHHE___,✗ FAIL
BBB,_IHE______,__________,__________,✓
CCC,_IHHE_IHE_,__________,___IE_____,✗ FAIL



*** FAILURES DETECTED ***


,Input,Expected,Got,Pass
Symbol,,,,
AAA,_IHHHHE___,___IHHE___,___IHHE___,✓
BBB,_IHE______,__________,__________,✓
CCC,_IHHE_IHE_,___IE_____,___IE_____,✓



All multi-symbol cases pass


## 5. Post-Offset Invariants

Properties that must hold for any **non-trivial** offset (`k ≥ 1`):

1. Every `enter == 1` bar has `signal_open == 0` and `cycle == "init"`.  The
   function explicitly forces `signal_open=0` on every new init it creates,
   and any original entries it preserves get wiped to flat (their bar 0 of the
   run becomes `None`).
2. Every `cycle == "None"` bar has `enter = exit = signal_open = 0`.
3. Every surviving run is a well-formed `init [held]* [exit]?` sequence.

`k=0` is a no-op (the function returns the input unchanged) so input invariants
pass through and aren't tested here.

In [ ]:
import re

VALID_RUN = re.compile(r"IH*E?")  # init, held*, optional exit


def check_invariants(df: pd.DataFrame) -> list[str]:
    errs = []
    enters = df[df["enter"] == 1]
    if not (enters["signal_open"] == 0).all():
        errs.append("enter==1 bar has signal_open != 0")
    if not (enters["cycle"] == "init").all():
        errs.append("enter==1 bar has cycle != 'init'")
    flats = df[df["cycle"] == "None"]
    if not (flats[["enter", "exit", "signal_open"]] == 0).all().all():
        errs.append("cycle=='None' bar has non-zero enter/exit/signal_open")
    for sym in df.index.get_level_values("symbol").unique():
        s = cycle_string(df, sym).replace("_", " ")
        for run in s.split():
            if not VALID_RUN.fullmatch(run):
                errs.append(f"{sym}: malformed run {run!r}")
    return errs


test_patterns = [
    {"S": "_IHHHE___"},
    {"S": "_IHHE_IHE___"},
    {"S": "IHHEIHE___"},
    {"A": "_IHHHHE___", "B": "_IHE______", "C": "_IHHE_IHE_"},
]

all_pass = True
for pats in test_patterns:
    df_in = make_signal_df(pats)
    for k in [1, 2, 3, 5, 10]:
        df_out = _apply_entry_offset(df_in, k)
        errs = check_invariants(df_out)
        if errs:
            all_pass = False
            print(f"FAIL  patterns={pats}  k={k}")
            for e in errs:
                print(f"        {e}")

print("All invariants hold" if all_pass else "*** INVARIANT VIOLATIONS ***")

## 6. Real-Data Sanity Check

Run `BarBacktest` on real bars with and without an offset and verify:

- Every surviving delayed entry timestamp is exactly `k` bars after the
  original entry on the same symbol.
- The number of trades doesn't increase under the offset.
- Every original trade with `duration > k+1` shows up as a surviving delayed
  trade.

In [11]:
from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalTradePerformance

signal  = TrendSignal(ma_window=200)
yahoo   = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

bars      = yahoo.get_bars(symbols, start=start - pd.offsets.BDay(signal.warmup),
                            end=end, adjust=False)
signal_df = signal.run(bars, trim_start=start)

K = 5
base    = BarBacktest().run(signal_df)
delayed = BarBacktest().run(signal_df, entry_offset=K)

base_trades    = SignalTradePerformance(base).trade_stats()
delayed_trades = SignalTradePerformance(delayed).trade_stats()

print(f"Base trades:    {len(base_trades)}")
print(f"Delayed trades: {len(delayed_trades)}  (entry_offset={K})")
print(f"Dropped:        {len(base_trades) - len(delayed_trades)}")
assert len(delayed_trades) <= len(base_trades), "offset should not create new trades"

# Each delayed entry should map back to a base entry exactly K bdays earlier.
mismatches = []
for _, row in delayed_trades.iterrows():
    sym = row["symbol"]
    delayed_entry = row["entry_date"]
    base_grp = base_trades[base_trades["symbol"] == sym]
    diffs = (delayed_entry - base_grp["entry_date"]).dt.days
    # Look for a base trade whose entry is 5 bdays earlier (allow 7 calendar days for weekends)
    candidates = base_grp[(diffs >= K) & (diffs <= K + 3)]
    if candidates.empty:
        mismatches.append({"symbol": sym, "delayed_entry": delayed_entry})

if mismatches:
    print(f"\n*** {len(mismatches)} delayed entries with no matching base entry ***")
    display(pd.DataFrame(mismatches))
else:
    print("Every delayed entry maps back to a base entry K bars earlier ✓")

# Every base trade with duration > K+1 must have survived
long_base = base_trades[base_trades["duration"] > K + 1]
missing = []
for _, row in long_base.iterrows():
    sym = row["symbol"]
    base_entry = row["entry_date"]
    delayed_grp = delayed_trades[delayed_trades["symbol"] == sym]
    diffs = (delayed_grp["entry_date"] - base_entry).dt.days
    if not ((diffs >= K) & (diffs <= K + 3)).any():
        missing.append({"symbol": sym, "base_entry": base_entry,
                        "duration": row["duration"]})

if missing:
    print(f"\n*** {len(missing)} long base trades with no surviving delayed counterpart ***")
    display(pd.DataFrame(missing))
else:
    print(f"All {len(long_base)} base trades with duration > {K+1} survive the offset ✓")

2026-05-04 01:08:38.011 | DEBUG    | hailmary.data.cache:get:34 - Cache expired for key=de1ec4900549
2026-05-04 01:08:38.014 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 3 symbols from Yahoo (2021-03-29 00:00:00 → 2024-01-01 00:00:00); inclusive
2026-05-04 01:08:38.464 | DEBUG    | hailmary.data.cache:set:45 - Cached 3027 rows key=de1ec4900549


Base trades:    17
Delayed trades: 10  (entry_offset=5)
Dropped:        7
Every delayed entry maps back to a base entry K bars earlier ✓
All 10 base trades with duration > 6 survive the offset ✓


2026-05-04 01:08:38.809 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=de1ec4900549


Base trades:    17
Delayed trades: 10  (entry_offset=5)
Dropped:        7
Every delayed entry maps back to a base entry K bars earlier ✓
All 10 base trades with duration > 6 survive the offset ✓


## 7. Visualise One Real Trade Under the Offset

Pick the longest base trade and overlay the original entry, the delayed entry,
and the shared exit on the price series.

In [ ]:
longest = base_trades.sort_values("duration", ascending=False).iloc[0]
sym = longest["symbol"]
base_entry  = longest["entry_date"]
exit_date   = longest["exit_date"]

sym_index = signal_df.loc[sym].index
delayed_entry = sym_index[sym_index.get_loc(base_entry) + K]

window_start = base_entry - pd.offsets.BDay(5)
window_end   = exit_date + pd.offsets.BDay(5)
px = base.data.loc[sym].loc[window_start:window_end, "close"]


def add_marker_line(fig, x, color, label, dash="solid"):
    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=x, x1=x, y0=0, y1=1,
        line=dict(color=color, width=2, dash=dash),
    )
    fig.add_annotation(
        x=x, y=1, xref="x", yref="paper",
        text=label, showarrow=False,
        font=dict(color=color, size=11),
        yshift=10, xanchor="left",
    )


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=px.index, y=px.values,
    name="close", line=dict(color=PALETTE["text_secondary"], width=1.5),
))
add_marker_line(fig, base_entry,    PALETTE["accent_blue"],   "base entry",   dash="dot")
add_marker_line(fig, delayed_entry, PALETTE["accent_orange"], f"delayed (+{K})")
add_marker_line(fig, exit_date,     PALETTE["accent_red"],    "exit",         dash="dash")

apply_theme(fig, title=f"{sym}: longest base trade with delayed entry overlay", height=400)
fig.update_layout(yaxis=dict(title_text="close"))
fig.show()

print(f"Symbol:         {sym}")
print(f"Base entry:     {base_entry.date()}")
print(f"Delayed entry:  {delayed_entry.date()}  (+{K} bars)")
print(f"Exit:           {exit_date.date()}")
print(f"Base duration:  {longest['duration']} bars")